# Using Feature Distillation to Augment DataSets and Enhance Performance Metrics (PyTorch Conversion)

### Dr. Ali Arsanjani, June 2019 (Converted by Gemini)

In this paper we describe a simple empirical technique to increase SOTA accuracy of slim datasets by up to 17%.
...

We demonstrate that with feature distillation we can enhance the metrics of this dataset, and generalize to other datasets.


## Preparation

First, let's import necessary modules.

In [43]:
import random, math
import time
import numpy as np
import pandas as pd
import collections
from collections import Counter

# PyTorch imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence

# Transformers imports
from transformers import BertModel, BertTokenizer, BertConfig

# Set seeds for reproducibility
np.random.seed(100)
torch.manual_seed(100)
random.seed(100)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(100)

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

bert_dropout = 0.1 # From later in the notebook
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
config = BertConfig.from_pretrained('bert-base-uncased',
                                    hidden_dropout_prob=bert_dropout,
                                    attention_probs_dropout_prob=bert_dropout)

Using device: cuda


In [44]:
bert_base = BertModel.from_pretrained('bert-base-uncased', config=config)
bert_base.to(device)
print(bert_base)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [45]:
class CustomVocab:
    def __init__(self, counter, min_freq=1, unknown_token='<unk>'):
        self.unknown_token = unknown_token
        self._token_to_idx = {unknown_token: 0}
        self._idx_to_token = [unknown_token]

        idx = 1
        for token, freq in counter.items():
            if freq >= min_freq:
                if token not in self._token_to_idx:
                    self._token_to_idx[token] = idx
                    self._idx_to_token.append(token)
                    idx += 1
        self._unknown_idx = self._token_to_idx[unknown_token]

    def __getitem__(self, token):
        return self._token_to_idx.get(token, self._unknown_idx)

    def __len__(self):
        return len(self._idx_to_token)

    def __repr__(self):
        return f"CustomVocab(size={len(self)})"

def load_tsv_data(filepath):
    # The original code used field_indices=[1, 2, ..., 13]
    # This implies skipping the first column (index 0) and reading the next 13.
    try:
        data = pd.read_csv(filepath,
                           sep='\t',
                           header=None,
                        #    usecols=range(1, 14),
                           on_bad_lines='skip',
                           encoding='utf-8')
        # Convert to list of lists to match gluonnlp.data.TSVDataset output
        return data.values.tolist()
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return []

train_dataset_raw = load_tsv_data('data/liar-plus/train2.tsv')
print("training set example:", train_dataset_raw[1])
validate_dataset_raw = load_tsv_data('data/liar-plus/val2.tsv')
print("validation set example:", validate_dataset_raw[1])
test_dataset_raw = load_tsv_data('data/liar-plus/test2.tsv')
print("test set example:", test_dataset_raw[1])

training set example: [1.0, '10540.json', 'half-true', 'When did the decline of coal start? It started when natural gas took off that started to begin in (President George W.) Bushs administration.', 'energy,history,job-accomplishments', 'scott-surovell', 'State delegate', 'Virginia', 'democrat', 0.0, 0.0, 1.0, 1.0, 0.0, 'a floor speech.', 'Surovell said the decline of coal "started when natural gas took off  That started to begin in President (George W. ) Bushs administration. "No doubt, natural gas has been gaining ground on coal in generating electricity. The trend started in the 1990s but clearly gained speed during the Bush administration when the production of natural gas -- a competitor of coal -- picked up. But analysts give little credit or blame to Bush for that trend. They note that other factors, such as technological innovation, entrepreneurship and policies of previous administrations, had more to do with laying the groundwork for the natural gas boom.']
validation set ex

In [46]:
for ele in validate_dataset_raw:
    print(ele[2:16])
    break

['barely-true', 'We have less Americans working now than in the 70s.', 'economy,jobs', 'vicky-hartzler', 'U.S. Representative', 'Missouri', 'republican', 1, 0, 1, 0, 0, 'an interview with ABC17 News', 'However, Hartzler was talking about the entire decade of the 70s. The first eight years of the decade, 1970 through 1977, have a lower employment-population ratio and a lower labor force participation rate than 2015.']


In [47]:
def analyze_dataset(raw_data):
    print("analyzing raw data set...\n <><><><><><><><><><> \n")

    label_cnt = Counter()
    topic_cnt = Counter()
    author_cnt = Counter()
    job_cnt = Counter()
    location_cnt = Counter()
    affiliation_cnt = Counter()

    # Try to extract the valid labels, topics, authors,
    for ele in raw_data:
        # Ensure element has enough columns
        if len(ele) < 13:
            continue

        try:
            label, statement, topics, author, job, location, affiliation,cnt_barely, cnt_false, cnt_half, cnt_mostly, cnt_pants_on_fire, venue_context, justification = [str(e) for e in ele[2:16]]

            label_cnt[label] += 1
            for topic in topics.lower().split(','):
                topic_cnt[topic.strip()] += 1
            author_cnt[author.lower()] += 1
            job_cnt[job.lower()] += 1
            location_cnt[location.lower()] += 1
            affiliation_cnt[affiliation.lower()] += 1
        except Exception as e:
            print(f"Error processing row: {ele} | Error: {e}")

    label_map = {ele: i for i, ele in enumerate(label_cnt.keys())}
    print("label map:",label_map)

    #!consider tuning the min_freq param
    # Replace gluonnlp.Vocab with CustomVocab
    topic_vocab = CustomVocab(topic_cnt, min_freq=100)
    author_vocab = CustomVocab(author_cnt, min_freq=50)
    job_vocab = CustomVocab(job_cnt, min_freq=50)
    location_vocab = CustomVocab(location_cnt, min_freq=50)
    affiliation_vocab = CustomVocab(affiliation_cnt, min_freq=50)

    print(topic_vocab)
    print(author_vocab)
    print(job_vocab)
    print(location_vocab)
    print(affiliation_vocab)
    return topic_vocab, author_vocab, job_vocab, location_vocab, affiliation_vocab, label_map

# Run analysis to get vocabs and label_map
topic_vocab, author_vocab, job_vocab, location_vocab, affiliation_vocab, label_map = analyze_dataset(train_dataset_raw)


def feature_extraction_transform(data):
    # Transform label into position / negative
    try:
        label, statement, topics, author, job, location, affiliation,cnt_barely, cnt_false, cnt_half, cnt_mostly, cnt_pants_on_fire, venue_context, justification = [str(e) for e in data[2:16]]
    except Exception as e:
        print(f"Error in feature_extraction_transform: {e}, data: {data}")
        return None # Return None to filter out this bad data

    topic_one_encoding = np.zeros(shape=(len(topic_vocab,)), dtype=np.float32)
    topic_ids = [topic_vocab[t.strip()] for t in topics.lower().split(',')]
    topic_one_encoding[topic_ids] = 1
    if len(topic_ids) > 0 and topic_one_encoding.sum() > 0:
        topic_one_encoding /= topic_one_encoding.sum()

    author_id = author_vocab[author.lower()]
    job_id = job_vocab[job.lower()]
    location_id = location_vocab[location.lower()]
    affiliation_id = affiliation_vocab[affiliation.lower()]

    try:
        cnt_barely_f = float(cnt_barely)
        cnt_false_f = float(cnt_false)
        cnt_half_f = float(cnt_half)
        cnt_mostly_f = float(cnt_mostly)
        cnt_pants_on_fire_f = float(cnt_pants_on_fire)
    except ValueError:
        # Handle cases where conversion to float fails
        cnt_barely_f = cnt_false_f = cnt_half_f = cnt_mostly_f = cnt_pants_on_fire_f = 0.0

    cnt_total = cnt_barely_f + cnt_false_f + cnt_half_f + cnt_mostly_f + cnt_pants_on_fire_f

    if cnt_total > 0 :
        proportion = [cnt_barely_f / cnt_total,
                      cnt_false_f / cnt_total,
                      cnt_half_f / cnt_total,
                      cnt_mostly_f / cnt_total,
                      cnt_pants_on_fire_f / cnt_total]
    else:
        proportion = [0.0, 0.0, 0.0, 0.0, 0.0]

    cnt_uncertainty = 1.0 / (cnt_total + 1.0)
    history_of_truth = np.array(proportion + [cnt_uncertainty], dtype=np.float32)
    venue_feature = 0 # venue_feature = f(venue_context), keep it as a vector

    return (statement, topic_one_encoding, author_id, job_id, location_id, affiliation_id,
            history_of_truth, venue_feature, label_map[label])

# Apply the first transform (feature extraction)
# .transform is replaced by a list comprehension
# Filter out None values from bad data
train_dataset = [d for d in [feature_extraction_transform(data) for data in train_dataset_raw] if d is not None]
validate_dataset = [d for d in [feature_extraction_transform(data) for data in validate_dataset_raw] if d is not None]
test_dataset = [d for d in [feature_extraction_transform(data) for data in test_dataset_raw] if d is not None]

print("train_dataset example:",train_dataset[0])
print('Original Train/Val/Test Size:', len(train_dataset), len(validate_dataset), len(test_dataset))

# Try to resplit the train/val dataset
# Replaces SimpleDataset with standard Python lists
all_train_val_samples = train_dataset + validate_dataset
all_idx = np.random.permutation(len(all_train_val_samples))
valid_ratio = 0.1
valid_num = int(valid_ratio * len(all_idx))
valid_idx = all_idx[:valid_num]
train_idx = all_idx[valid_num:]
train_dataset = [all_train_val_samples[ele] for ele in train_idx]
validate_dataset = [all_train_val_samples[ele] for ele in valid_idx]
print('Resplit Train/Val/Test Size:', len(train_dataset), len(validate_dataset), len(test_dataset))

analyzing raw data set...
 <><><><><><><><><><> 

label map: {'false': 0, 'half-true': 1, 'mostly-true': 2, 'true': 3, 'barely-true': 4, 'pants-fire': 5, 'nan': 6}
CustomVocab(size=60)
CustomVocab(size=22)
CustomVocab(size=22)
CustomVocab(size=28)
CustomVocab(size=7)
train_dataset example: ('Says the Annies List political group supports third-trimester abortions on demand.', array([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32), 0, 1, 1, 1, array([0. , 1. , 0. , 0. , 0. , 0.5], dtype=float32), 0, 0)
Original Train/Val/Test Size: 10242 1284 1267
Resplit Train/Val/Test Size: 10374 1152 1267


In [48]:
class BERTClassifier(nn.Module):
    def __init__(self, bert, num_topics, num_authors, num_jobs, num_locations, num_affiliations, num_classes,
                 embed_dim,
                 author_dropout, author_mlp_layers, author_mlp_hidden,
                 history_dropout, history_mlp_layers, history_mlp_hidden):
        super(BERTClassifier, self).__init__()
        self.bert = bert

        # Note: BERT hidden size is 768
        bert_hidden_size = bert.config.hidden_size

        self.topic_embed = nn.Linear(num_topics, embed_dim) # Use one-hot encoding for topics
        self.author_embed = nn.Embedding(num_embeddings=num_authors, embedding_dim=embed_dim)
        self.job_embed = nn.Embedding(num_embeddings=num_jobs, embedding_dim=embed_dim)
        self.location_embed = nn.Embedding(num_embeddings=num_locations, embedding_dim=embed_dim)
        self.affiliation_embed = nn.Embedding(num_embeddings=num_affiliations, embedding_dim=embed_dim)

        author_feature_map_layers = []
        author_feature_map_layers.append(nn.Dropout(author_dropout))
        author_input_dim = embed_dim * 5 # topic + author + job + location + affiliation
        for _ in range(author_mlp_layers):
            author_feature_map_layers.append(nn.Linear(author_input_dim, author_mlp_hidden))
            author_feature_map_layers.append(nn.LeakyReLU(0.1))
            author_feature_map_layers.append(nn.Dropout(author_dropout))
            author_input_dim = author_mlp_hidden # Input for next layer
        self.author_feature_map = nn.Sequential(*author_feature_map_layers)

        history_feature_map_layers = []
        history_input_dim = 6 # 5 proportions + 1 uncertainty
        for _ in range(history_mlp_layers):
            history_feature_map_layers.append(nn.Linear(history_input_dim, history_mlp_hidden))
            history_feature_map_layers.append(nn.LeakyReLU(0.1))
            history_feature_map_layers.append(nn.Dropout(history_dropout))
            history_input_dim = history_mlp_hidden # Input for next layer
        self.history_feature_map = nn.Sequential(*history_feature_map_layers)

        # extra layer used for classification
        classifier_input_dim = bert_hidden_size + author_input_dim + history_input_dim
        self.classifier = nn.Linear(classifier_input_dim, num_classes)


    def forward(self, inputs, segment_types, attention_mask,
                topic_one_hot, author_id, job_id, location_id, affiliation_id, history_feature):

        # Encode the news representation using BERT
        # seq_len is replaced by attention_mask
        outputs = self.bert(input_ids=inputs,
                            token_type_ids=segment_types,
                            attention_mask=attention_mask)

        # We use the pooler_output, which corresponds to the [CLS] token
        cls_encoding = outputs.pooler_output

        #dataset specific features:
        topic_fea = self.topic_embed(topic_one_hot)
        author_fea = self.author_embed(author_id)
        job_fea = self.job_embed(job_id)
        location_fea = self.location_embed(location_id)
        affiliation_fea = self.affiliation_embed(affiliation_id)

        # Concat author-related features
        author_features_combined = torch.cat((topic_fea, author_fea, job_fea, location_fea, affiliation_fea), dim=-1)
        author_feature = self.author_feature_map(author_features_combined)

        history_feature = self.history_feature_map(history_feature)

        # Concat all features for final classification
        combined_features = torch.cat((cls_encoding, author_feature, history_feature), dim=-1)

        return self.classifier(combined_features)

In [49]:
### BERT-specific Transformations
# Replaces gluonnlp.data.BERTSentenceTransform
# This will be applied *per item* before batching

padding_id = tokenizer.pad_token_id
print(f"Padding token ID: {padding_id}")

def transform_fn(text, topic_one_encoding, author_id, job_id, location_id, affiliation_id, history_feature, venue_ids, label):
    max_len = 256

    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,  # Adds [CLS] and [SEP]
        max_length=max_len,
        truncation=True,
        padding=False,            # We will pad in the collate_fn
        return_token_type_ids=True
    )

    data = np.array(encoding['input_ids'], dtype='int64')
    length = np.array(len(data), dtype='int64') # This is now just the length, not a tensor
    segment_type = np.array(encoding['token_type_ids'], dtype='int64')

    # history_feature is already a numpy array, just ensure type
    history_feature = np.array(history_feature, dtype=np.float32)

    # Return all features as standard types for batching
    return (data, length, segment_type, topic_one_encoding, author_id, job_id,
            location_id, affiliation_id, history_feature, label)

Padding token ID: 0


In [50]:
# Apply the BERT transform
# .transform(lazy=False) is replaced by a list comprehension
print("Applying BERT transform to train_dataset...")
train_dataset_bert = [transform_fn(*sample) for sample in train_dataset]
print("Applying BERT transform to validate_dataset...")
validate_dataset_bert = [transform_fn(*sample) for sample in validate_dataset]
print("Applying BERT transform to test_dataset...")
test_dataset_bert = [transform_fn(*sample) for sample in test_dataset]

print("Example transformed data:", train_dataset_bert[0])

Applying BERT transform to train_dataset...
Applying BERT transform to validate_dataset...
Applying BERT transform to test_dataset...
Example transformed data: (array([  101,  7402,  2015,  1999,  7756,  2490,  4547, 10995,  6115,
        1010,  5094,  2062,  2084, 10332,  1010,  2199,  2493,  3229,
        2488,  2816,  1012,  2021,  6615,  2522, 19731,  2480, 15429,
        2080,  2003,  2114,  2009,  1012,   102], dtype=int64), array(33, dtype=int64), array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int64), array([0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32), 0, 3, 4, 4, array([0.        , 0.        , 0.5       , 0.5       , 0.        ,
       0.33333334], dtype=float32), 2)


### Custom Dataset Class

In [51]:
class ListDataset(Dataset):
    def __init__(self, data_list):
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

train_data_torch = ListDataset(train_dataset_bert)
validate_data_torch = ListDataset(validate_dataset_bert)
test_data_torch = ListDataset(test_dataset_bert)


### Batchify (Collate) Function
Replaces `nlp.data.batchify`
This function will be passed to the DataLoader to create batches.

In [52]:
def collate_fn(batch):
    (data, lengths, segment_type, topic_one_encoding, author_id, job_id,
     location_id, affiliation_id, history_feature, label) = zip(*batch)

    # Pad sequences
    data_padded = pad_sequence(
        [torch.as_tensor(d, dtype=torch.long) for d in data],
        batch_first=True, padding_value=padding_id
    )

    seg_types_padded = pad_sequence(
        [torch.as_tensor(s, dtype=torch.long) for s in segment_type],
        batch_first=True, padding_value=0
    )

    attention_mask = (data_padded != padding_id).long()

    # Convert to tensors (no stacking needed if already numbers)
    lengths = torch.tensor([int(l) for l in lengths], dtype=torch.long)
    topic_one_encoding = torch.as_tensor(topic_one_encoding, dtype=torch.float32)
    author_id = torch.as_tensor(author_id, dtype=torch.long)
    job_id = torch.as_tensor(job_id, dtype=torch.long)
    location_id = torch.as_tensor(location_id, dtype=torch.long)
    affiliation_id = torch.as_tensor(affiliation_id, dtype=torch.long)
    history_feature = torch.as_tensor(history_feature, dtype=torch.float32)
    label = torch.as_tensor(label, dtype=torch.long)

    # Optional pinning (usually safe to skip unless you explicitly use pin_memory=True in DataLoader)
    for t in [data_padded, seg_types_padded, attention_mask, lengths, topic_one_encoding,
              author_id, job_id, location_id, affiliation_id, history_feature, label]:
        t = t.pin_memory()

    return (data_padded, seg_types_padded, attention_mask,
            topic_one_encoding, author_id, job_id, location_id,
            affiliation_id, history_feature, label)



# Create DataLoaders
# Replaces FixedBucketSampler with standard DataLoader + collate_fn
batch_size = 16

train_data = DataLoader(train_data_torch,
                        batch_size=batch_size,
                        shuffle=True,
                        collate_fn=collate_fn,
                        num_workers=0,      # 0 in notebooks on Windows; try 2-8 in scripts
                        pin_memory=False)

validate_data = DataLoader(validate_data_torch,
                           batch_size=batch_size,
                           shuffle=False,
                           collate_fn=collate_fn,
                           num_workers=0,
                           pin_memory=False)

test_data = DataLoader(test_data_torch,
                       batch_size=batch_size,
                       shuffle=False,
                       collate_fn=collate_fn,
                       num_workers=0,
                       pin_memory=False)


### Training Loop (Rewritten for PyTorch)

In [ ]:
import time
import torch

def evaluate_loop(net, eval_data, device):
    net.eval()
    total_num = 0
    hits = 0
    running_loss = 0.0

    with torch.no_grad():
        for batch_idx, (inputs, token_types, attention_mask, topic_one_encoding,
                        author_id, job_id, location_id, affiliation_id, history, label) in enumerate(eval_data):

            # Move tensors to device
            inputs = inputs.to(device, non_blocking=True)
            token_types = token_types.to(device, non_blocking=True)
            attention_mask = attention_mask.to(device, non_blocking=True)
            label = label.to(device, non_blocking=True)
            topic_one_encoding = topic_one_encoding.to(device, non_blocking=True)
            author_id = author_id.to(device, non_blocking=True)
            job_id = job_id.to(device, non_blocking=True)
            location_id = location_id.to(device, non_blocking=True)
            affiliation_id = affiliation_id.to(device, non_blocking=True)
            history = history.to(device, non_blocking=True)

            out = net(inputs, token_types, attention_mask, topic_one_encoding,
                      author_id, job_id, location_id, affiliation_id, history)

            total_num += out.shape[0]
            hits += (out.argmax(dim=-1) == label).sum().item()

    return hits / float(total_num)


scaler = torch.amp.GradScaler()
torch.backends.cudnn.benchmark = True


def train_loop(net, train_data, test_data, validate_data,
               num_epoch, lr, wd, optimizer, device, loss_fn):

    best_val_acc = 0.0
    best_test_acc = 0.0
    best_epoch = 0

    print("🚀 Starting training...\n")

    for epoch in range(num_epoch):
        start_time = time.time()
        net.train()
        running_loss = 0.0
        total_correct = 0
        total_samples = 0

        print(f"\n🔹 Epoch {epoch+1}/{num_epoch}")
        print("-" * 60)

        for i, (inputs, token_types, attention_mask, topic_one_encoding,
                author_id, job_id, location_id, affiliation_id, history, label) in enumerate(train_data):

            # Move data to device
            inputs = inputs.to(device, non_blocking=True)
            token_types = token_types.to(device, non_blocking=True)
            attention_mask = attention_mask.to(device, non_blocking=True)
            label = label.to(device, non_blocking=True)
            topic_one_encoding = topic_one_encoding.to(device, non_blocking=True)
            author_id = author_id.to(device, non_blocking=True)
            job_id = job_id.to(device, non_blocking=True)
            location_id = location_id.to(device, non_blocking=True)
            affiliation_id = affiliation_id.to(device, non_blocking=True)
            history = history.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            # Forward + backward under mixed precision
            with torch.amp.autocast(device_type='cuda'):
                out = net(inputs, token_types, attention_mask, topic_one_encoding,
                          author_id, job_id, location_id, affiliation_id, history)
                loss = loss_fn(out, label)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            total_correct += (out.argmax(dim=-1) == label).sum().item()
            total_samples += label.size(0)

            # Batch log
            if i % 100 == 0 or i == len(train_data) - 1:
                acc = 100 * total_correct / total_samples
                avg_loss = running_loss / (i + 1)
                mem_alloc = torch.cuda.memory_allocated(device) / 1024**2
                mem_reserved = torch.cuda.memory_reserved(device) / 1024**2
                print(f"Batch {i:4d} | "
                      f"Loss: {avg_loss:.4f} | "
                      f"Acc: {acc:.2f}% | "
                      f"Mem: {mem_alloc:.1f}/{mem_reserved:.1f} MB")

        # Evaluate after each epoch
        val_acc = evaluate_loop(net, validate_data, device)
        test_acc = evaluate_loop(net, test_data, device)
        elapsed = time.time() - start_time

        print(f"\n✅ Epoch {epoch+1} complete "
              f"({elapsed:.1f}s): Train Acc={100*total_correct/total_samples:.2f}%, "
              f"Val Acc={100*val_acc:.2f}%, Test Acc={100*test_acc:.2f}%")

        # Save model if improved
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_test_acc = test_acc
            best_epoch = epoch
            torch.save(net.state_dict(), './checkpoints/bert_liar_liar_best.pth')
            print(f"💾 New best model saved (epoch {epoch+1}) "
                  f"with val_acc={best_val_acc:.4f}")

    print("\n🎯 Training complete.")
    print(f"Best Epoch: {best_epoch+1} | "
          f"Val Acc: {100*best_val_acc:.2f}% | "
          f"Test Acc: {100*best_test_acc:.2f}%")

    return best_val_acc, best_test_acc, best_epoch

In [54]:
#<><><><><><><><><><><><><><><><><><><><><><>
#<><> Training Params and Main Loop
#<><><><><><><><><><><><><><><><><><><><><><>
num_epoch = 21
lr = 0.0001
wd = 1E-4
warm_up = 5000


# Network parameters
embed_dim = 16
author_dropout = 0.1
author_mlp_layers = 2
author_mlp_hidden = 128
history_dropout = 0.1
history_mlp_layers = 2
history_mlp_hidden = 128

# Model, Loss, and Optimizer
net = BERTClassifier(
    bert=bert_base,
    num_topics=len(topic_vocab),
    num_authors=len(author_vocab),
    num_jobs=len(job_vocab),
    num_locations=len(location_vocab),
    num_affiliations=len(affiliation_vocab),
    num_classes=len(label_map),
    embed_dim=embed_dim,
    author_dropout=author_dropout,
    author_mlp_layers=author_mlp_layers,
    author_mlp_hidden=author_mlp_hidden,
    history_dropout=history_dropout,
    history_mlp_layers=history_mlp_layers,
    history_mlp_hidden=history_mlp_hidden)

net.to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=wd)

acc_val = [0] * 100
acc_test = [0] * 100
loss_val = [0] * 100

timestamp = time.strftime('-%Y-%m-%d-%H-%M-%S', time.gmtime())
print(timestamp)

best_val_acc, best_test_acc, best_epoch = train_loop(
    net, train_data, test_data, validate_data,
    num_epoch, lr, wd, optimizer, device, loss_fn
)

timestamp = time.strftime('-%Y-%m-%d-%H-%M-%S', time.gmtime())
print(timestamp,"best_val_acc: {}, best_test_acc:{}, best_epoch:{}".format(best_val_acc, best_test_acc, best_epoch))

-2025-11-07-22-28-10
🚀 Starting training...


🔹 Epoch 1/21
------------------------------------------------------------
Batch    0 | Loss: 1.8704 | Acc: 25.00% | Mem: 3382.0/4232.0 MB
Batch  100 | Loss: nan | Acc: 19.43% | Mem: 3390.7/5140.0 MB
Batch  200 | Loss: nan | Acc: 20.93% | Mem: 3381.6/5140.0 MB
Batch  300 | Loss: nan | Acc: 22.43% | Mem: 3390.9/5140.0 MB
Batch  400 | Loss: nan | Acc: 25.14% | Mem: 3391.4/5140.0 MB
Batch  500 | Loss: nan | Acc: 27.16% | Mem: 3391.4/5140.0 MB
Batch  600 | Loss: nan | Acc: 29.37% | Mem: 3381.6/5140.0 MB
Batch  648 | Loss: nan | Acc: 30.31% | Mem: 3387.5/5140.0 MB

✅ Epoch 1 complete (53.5s): Train Acc=30.31%, Val Acc=44.88%, Test Acc=43.49%
💾 New best model saved (epoch 1) with val_acc=0.4488

🔹 Epoch 2/21
------------------------------------------------------------
Batch    0 | Loss: 1.6796 | Acc: 31.25% | Mem: 3388.7/5188.0 MB
Batch  100 | Loss: 1.4889 | Acc: 44.12% | Mem: 3393.1/5278.0 MB
Batch  200 | Loss: 1.4755 | Acc: 43.31% | Mem: 3395.8/

### Prediction
This section is updated to load the saved PyTorch model state.

In [57]:
#<><><><><><><><><><><><><><><><><><><><><><><><><><><><><><><><><>
#??? Provide a document to predict level of fakeness

# Create a new instance of the model
net_best = BERTClassifier(
    bert=bert_base,
    num_topics=len(topic_vocab),
    num_authors=len(author_vocab),
    num_jobs=len(job_vocab),
    num_locations=len(location_vocab),
    num_affiliations=len(affiliation_vocab),
    num_classes=len(label_map),
    embed_dim=embed_dim,
    author_dropout=author_dropout,
    author_mlp_layers=author_mlp_layers,
    author_mlp_hidden=author_mlp_hidden,
    history_dropout=history_dropout,
    history_mlp_layers=history_mlp_layers,
    history_mlp_hidden=history_mlp_hidden)

# Load the saved state dictionary
net_best.load_state_dict(torch.load('./checkpoints/bert_liar_liar_best.pth'))
net_best.to(device) # Move to device

# Evaluate
test_acc = evaluate_loop(net_best, test_data, device)
print('Test acc =', test_acc)

Test acc = 0.4325177584846093
